In [ ]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q decord transformers accelerate timm torchvision \
                opencv-python pillow tqdm av imageio imageio-ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 23.8 MB/s eta 0:00:00


Video Preprocessing


In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# Imports
# ============================================================

import os
import cv2
import math
import torch
import random
import numpy as np
import pandas as pd

from tqdm import tqdm
from pathlib import Path
from PIL import Image

from decord import VideoReader
from decord import cpu

from transformers import VideoMAEImageProcessor

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# Configuration
# ============================================================

DATASET_ROOT = "/content/drive/MyDrive/sentinalMAe"

OUTPUT_ROOT = "/content/SentinelMAE_Processed"

CLASSES = [
    "Fighting",
    "Normal",
    "Shooting"
]

SPLITS = [
    "train",
    "test"
]

NUM_FRAMES = 16

IMAGE_SIZE = 224

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device :", DEVICE)

Device : cpu


In [ ]:
# ============================================================
# VideoMAE Processor
# ============================================================

processor = VideoMAEImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base"
)

print("VideoMAE Processor Loaded Successfully")

VideoMAE Processor Loaded Successfully


In [ ]:
# ============================================================
# Verify Dataset
# ============================================================

for split in SPLITS:

    print("="*60)
    print(split.upper())

    for cls in CLASSES:

        folder = Path(DATASET_ROOT)/split/cls

        videos = []

        for ext in ["*.mp4","*.avi","*.mov","*.mkv"]:

            videos.extend(folder.glob(ext))

        print(f"{cls:12s} : {len(videos)} videos")

TRAIN
Fighting     : 377 videos
Normal       : 2049 videos
Shooting     : 232 videos
TEST
Fighting     : 107 videos
Normal       : 300 videos
Shooting     : 62 videos


In [ ]:
# ============================================================
# Create Output Directories
# ============================================================

for split in SPLITS:

    for cls in CLASSES:

        video_dir = Path(
            OUTPUT_ROOT,
            split,
            cls,
            "video"
        )

        video_dir.mkdir(
            parents=True,
            exist_ok=True
        )

print("Folders Created Successfully")

Folders Created Successfully


In [ ]:
# ============================================================
# Uniform Frame Sampling
# ============================================================

def sample_frame_indices(num_frames, total_frames):

    if total_frames < num_frames:
        indices = np.linspace(
            0,
            total_frames - 1,
            total_frames,
            dtype=int
        )

        indices = list(indices)

        while len(indices) < num_frames:
            indices.append(indices[-1])

        return indices

    return np.linspace(
        0,
        total_frames - 1,
        num_frames,
        dtype=int
    )

In [ ]:
# ============================================================
# Read Video using Decord
# ============================================================

def read_video(video_path):

    vr = VideoReader(
        str(video_path),
        ctx=cpu(0)
    )

    total_frames = len(vr)

    indices = sample_frame_indices(
        NUM_FRAMES,
        total_frames
    )

    frames = vr.get_batch(indices).asnumpy()

    return frames

In [ ]:
# ============================================================
# VideoMAE Preprocessing
# ============================================================

def preprocess_video(video_path):

    frames = read_video(video_path)

    frames = [Image.fromarray(frame) for frame in frames]

    encoding = processor(
        frames,
        return_tensors="pt"
    )

    pixel_values = encoding["pixel_values"]

    pixel_values = pixel_values.squeeze(0)

    return pixel_values

In [ ]:
# ============================================================
# Test One Video
# ============================================================

video = list(
    (Path(DATASET_ROOT) /
     "train" /
     "Fighting").glob("*.mp4")
)[0]

tensor = preprocess_video(video)

print("Shape :", tensor.shape)

print("Dtype :", tensor.dtype)

Shape : torch.Size([16, 3, 224, 224])
Dtype : torch.float32


In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   37G   71G  35% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G  4.0K  5.7G   1% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
tmpfs           6.4G  272K  6.4G   1% /var/colab
/dev/sda1       114G   51G   64G  45% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive           100G   94G  6.4G  94% /content/drive


In [ ]:
# ============================================================
# Process Complete Dataset (Save to Colab)
# ============================================================

from pathlib import Path
from tqdm import tqdm
import torch

# Save in Colab local storage
OUTPUT_ROOT = "/content/SentinelMAE_Processed"

processed = 0
skipped = 0
failed = 0

failed_files = []

for split in SPLITS:

    print("\n" + "="*60)
    print(split.upper())
    print("="*60)

    for cls in CLASSES:

        print(f"\nProcessing {cls}...")

        input_folder = Path(DATASET_ROOT) / split / cls
        output_folder = Path(OUTPUT_ROOT) / split / cls / "video"

        output_folder.mkdir(parents=True, exist_ok=True)

        videos = []

        for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv"]:
            videos.extend(sorted(input_folder.glob(ext)))

        for idx, video_path in enumerate(tqdm(videos, desc=f"{split}/{cls}")):

            save_path = output_folder / f"{cls.lower()}_{idx+1:06d}.pt"

            # Skip if already processed
            if save_path.exists():
                skipped += 1
                continue

            try:
                tensor = preprocess_video(video_path)

                # Save as float16 to reduce storage by ~50%
                tensor = tensor.half()

                torch.save(
                    {
                        "video": tensor,
                        "label": cls,
                        "original_file": video_path.name
                    },
                    save_path
                )

                processed += 1

            except Exception as e:

                failed += 1
                failed_files.append(str(video_path))

                print(f"\nSkipping corrupted video: {video_path.name}")
                print("Reason:", e)

                continue

print("\n" + "="*60)
print("VIDEO PREPROCESSING COMPLETED")
print("="*60)

print(f"Processed : {processed}")
print(f"Skipped   : {skipped}")
print(f"Failed    : {failed}")

# Save failed files log
log_path = "/content/failed_videos.txt"

with open(log_path, "w") as f:
    for item in failed_files:
        f.write(item + "\n")

print(f"\nFailed video list saved to: {log_path}")


TRAIN

Processing Fighting...


train/Fighting: 100%|██████████| 377/377 [00:00<00:00, 30732.58it/s]



Processing Normal...


train/Normal:   0%|          | 0/2049 [00:00<?, ?it/s]


Skipping corrupted video: Normal_v=8cTqh9tMz_I___1_label_A.mp4
Reason: Error reading /content/drive/MyDrive/sentinalMAe/train/Normal/Normal_v=8cTqh9tMz_I___1_label_A.mp4...


...


Processing Normal...


test/Normal: 100%|██████████| 300/300 [18:08<00:00,  3.63s/it]



Processing Shooting...


test/Shooting: 100%|██████████| 62/62 [02:03<00:00,  2.00s/it]


VIDEO PREPROCESSING COMPLETED
Processed : 814
Skipped   : 2312
Failed    : 1

Failed video list saved to: /content/failed_videos.txt


In [ ]:
from pathlib import Path
import random
import torch

files = list(Path("/content/SentinelMAE_Processed").rglob("*.pt"))

print("Total processed:", len(files))

sample = random.choice(files)

data = torch.load(sample)

print("Tensor Shape :", data["video"].shape)
print("Tensor Type  :", data["video"].dtype)
print("Label        :", data["label"])
print("Original File:", data["original_file"])

Total processed: 3126
Tensor Shape : torch.Size([16, 3, 224, 224])
Tensor Type  : torch.float16
Label        : Normal
Original File: Normal_Young.And.Dangerous.IV.1997___0-10-35_0-11-50_label_A.mp4


In [ ]:
!du -sh /content/SentinelMAE_Processed

15G	/content/SentinelMAE_Processed


In [ ]:
from pathlib import Path

root = Path("/content/SentinelMAE_Processed")

for path in root.iterdir():
    print(path)

/content/SentinelMAE_Processed/train
/content/SentinelMAE_Processed/test


In [ ]:
!zip -r /content/SentinelMAE_Processed.zip /content/SentinelMAE_Processed

  adding: content/SentinelMAE_Processed/ (stored 0%)
  adding: content/SentinelMAE_Processed/train/ (stored 0%)
  adding: content/SentinelMAE_Processed/train/Normal/ (stored 0%)
  adding: content/SentinelMAE_Processed/train/Normal/video/ (stored 0%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_000803.pt (deflated 54%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_000522.pt (deflated 63%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_000332.pt (deflated 61%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_002005.pt (deflated 62%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_000145.pt (deflated 66%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_000042.pt (deflated 59%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_000298.pt (deflated 61%)
  adding: content/SentinelMAE_Processed/train/Normal/video/normal_000428.pt (deflated 61%)
  adding: content/Sent

In [ ]:
from google.colab import files

files.download("/content/SentinelMAE_Processed.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>